# 两湖结果分析、绘图与小体积证据打包

本notebook不运行模型、不改原始输入，也不修改已封存的结果。首先运行前两个代码单元，在RStudio执行打印的两行命令完成分析，然后运行最后的打包单元。

新结果尚未上传，因此这里提供的是分析代码，不能预先声称哪湖或哪模型更准确。报告包含四模型与Ensemble4的指标、配对窗口温度、残差季节性、最优参数及参数响应；参数散点不等同于δ/S1敏感性分析。


In [ ]:
PROJECT = "C:/Geography/R/dissertation_LakeEnsemblR"
ISO = "D:/Geography/R/dissertation_LakeEnsemblR/LER_window/MyLake_v121_ordering_isolated_20261029"
RUN_ROOT = "D:/LER4_ice_20261102/n500"
CONFIG_ROOT = PROJECT + "/yaml/r4_ice_20261102"


In [ ]:
from pathlib import Path
import os
import json

def local_path(value):
    text = str(value).replace("\\", "/")
    if os.name != "nt" and len(text) >= 3 and text[1:3] == ":/":
        text = "/mnt/" + text[0].lower() + text[2:]
    return Path(text)

R_ANALYSIS = '# Read-only analysis of completed four-model runs. No engines or model inputs are changed.\n.a4_need <- function(ok, ...) if (!isTRUE(ok)) stop(..., call. = FALSE)\n.a4_read <- function(p) {\n  .a4_need(file.exists(p), "Missing analysis input: ", p)\n  utils::read.csv(p, stringsAsFactors = FALSE, check.names = FALSE, fileEncoding = "UTF-8-BOM")\n}\n.a4_sha <- function(p) digest::digest(file = p, algo = "sha256", serialize = FALSE)\n.a4_csv <- function(x, p) utils::write.csv(x, p, row.names = FALSE, fileEncoding = "UTF-8", na = "NA")\n.a4_receipt <- function(p) {\n  .a4_need(file.exists(p) && file.exists(paste0(p, ".sha256")), "Missing receipt/seal: ", p)\n  .a4_need(identical(readLines(paste0(p, ".sha256"), warn = FALSE), .a4_sha(p)), "Receipt hash differs: ", p)\n  readRDS(p)\n}\n.a4_report_check <- function(lake_dir) {\n  record <- .a4_receipt(file.path(lake_dir, "report.rds"))\n  root <- file.path(lake_dir, "report")\n  expected <- record$files\n  current <- sort(list.files(root, recursive = TRUE, all.files = TRUE, no.. = TRUE))\n  .a4_need(!anyDuplicated(expected$file) && setequal(current, as.character(expected$file)), "Report file list changed: ", root)\n  for (i in seq_len(nrow(expected))) {\n    p <- file.path(root, expected$file[i])\n    .a4_need(file.info(p)$size == expected$bytes[i] && identical(.a4_sha(p), expected$sha256[i]),\n      "Report artifact changed: ", p)\n  }\n  invisible(record)\n}\n.a4_metrics <- function(x) {\n  e <- x$predicted - x$observed\n  data.frame(N = nrow(x), RMSE = sqrt(mean(e^2)), MAE = mean(abs(e)), bias = mean(e),\n    NSE = if (sum((x$observed - mean(x$observed))^2) > 0) 1 - sum(e^2)/sum((x$observed - mean(x$observed))^2) else NA_real_,\n    r = if (nrow(x) >= 2L && stats::sd(x$observed) > 0 && stats::sd(x$predicted) > 0) stats::cor(x$observed, x$predicted) else NA_real_)\n}\n.a4_figure <- function(out, name, draw, width = 1800, height = 1100) {\n  for (ext in c("png", "pdf")) {\n    p <- file.path(out, paste0(name, ".", ext))\n    if (ext == "png") grDevices::png(p, width = width, height = height, res = 160)\n    else grDevices::pdf(p, width = width/160, height = height/160, useDingbats = FALSE)\n    tryCatch(draw(), finally = grDevices::dev.off())\n  }\n}\n\nanalyze_two_lakes_r4 <- function(run_root = "D:/LER4_ice_20261102/n500", out_dir = NULL, old_akesaiqin_metrics = NULL) {\n  .a4_need(requireNamespace("digest", quietly = TRUE), "Load the original isolated R library containing digest first")\n  run_root <- normalizePath(run_root, winslash = "/", mustWork = TRUE)\n  lakes <- c("akesaiqin", "namco")\n  models <- c("FLake", "GLM", "GOTM", "Simstrat", "Ensemble4")\n  colors <- setNames(c("#3686A0", "#E3A437", "#4D9D70", "#A06FAB", "#444444"), models)\n  index <- .a4_read(file.path(run_root, "two_lake_run_index.csv"))\n  index <- index[match(lakes, index$name), , drop = FALSE]\n  .a4_need(identical(as.character(index$name), lakes) && all(index$calibration_status == "ok") &&\n    all(index$evaluation_status == "ok"), "Both lakes must have completed calibration and evaluation")\n  if (is.null(out_dir)) out_dir <- file.path(dirname(run_root), paste0("analysis_two_lakes_", format(Sys.time(), "%Y%m%dT%H%M%S")))\n  .a4_need(!file.exists(out_dir), "Use a NEW analysis output directory: ", out_dir)\n  .a4_need(dir.create(out_dir, recursive = TRUE), "Cannot create analysis output directory")\n  out_dir <- normalizePath(out_dir, winslash = "/", mustWork = TRUE)\n  metric_list <- pair_list <- param_list <- response_list <- timing_list <- list()\n  for (lake in lakes) {\n    root <- file.path(run_root, lake); report <- file.path(root, "report")\n    .a4_report_check(root)\n    metrics <- .a4_read(file.path(report, "model_metrics.csv"))\n    ensemble <- .a4_read(file.path(report, "ensemble4_metrics.csv"))\n    fields <- c("model", "split", "N", "RMSE", "MAE", "bias", "NSE", "r")\n    metrics <- rbind(metrics[fields], ensemble[fields])\n    .a4_need(nrow(metrics) == 10L && setequal(metrics$model, models), "Unexpected metrics/model design: ", lake)\n    pairs <- .a4_read(file.path(report, "pairs_models_ensemble4.csv"))\n    .a4_need(setequal(pairs$model, models) && all(is.finite(pairs$predicted)) && all(is.finite(pairs$observed)),\n      "Expected finite four-model plus ensemble window pairs: ", lake)\n    .a4_need(!anyDuplicated(paste(pairs$model, pairs$split, pairs$window_id)), "Duplicate model/split/window: ", lake)\n    pairs$start_UTC <- as.POSIXct(pairs$window_start, tz = "UTC")\n    pairs$end_UTC <- as.POSIXct(pairs$window_end, tz = "UTC")\n    .a4_need(all(!is.na(pairs$start_UTC) & !is.na(pairs$end_UTC) & pairs$end_UTC > pairs$start_UTC), "Invalid window dates")\n    pairs$midpoint_UTC <- pairs$start_UTC + as.numeric(difftime(pairs$end_UTC, pairs$start_UTC, units = "secs"))/2\n    pairs$month <- as.integer(format(pairs$midpoint_UTC, "%m", tz = "UTC"))\n    pairs$residual <- pairs$predicted - pairs$observed\n    for (i in seq_len(nrow(metrics))) {\n      z <- pairs[pairs$model == metrics$model[i] & pairs$split == metrics$split[i], , drop = FALSE]\n      calc <- .a4_metrics(z)\n      .a4_need(isTRUE(all.equal(as.numeric(unlist(metrics[i, names(calc)])), as.numeric(unlist(calc[1, ])), tolerance = 1e-9)),\n        "Reported metrics do not reproduce from equal-weight window pairs: ", lake, "/", metrics$model[i], "/", metrics$split[i])\n    }\n    for (split in c("calibration", "validation")) {\n      ref <- pairs[pairs$model == models[1] & pairs$split == split, , drop = FALSE]\n      for (model in models[-1]) {\n        z <- pairs[pairs$model == model & pairs$split == split, , drop = FALSE]\n        .a4_need(setequal(z$window_id, ref$window_id) && isTRUE(all.equal(z$observed[match(ref$window_id, z$window_id)], ref$observed)),\n          "Models do not share identical observed windows: ", lake)\n      }\n    }\n    parameters <- .a4_read(file.path(report, "selected_parameters.csv"))\n    .a4_need(nrow(parameters) == 9L && sum(parameters$model == "Simstrat") == 3L, "Expected 2+2+2+3 selected parameters")\n    parameters$lower <- ifelse(parameters$parameter == "ModelParameters/p_sw_ice", 0.9, 0.5)\n    parameters$upper <- ifelse(parameters$parameter == "wind_speed", 2, ifelse(parameters$parameter == "swr", 1.5, 3))\n    parameters$fraction_of_range <- (parameters$value - parameters$lower)/(parameters$upper - parameters$lower)\n    parameters$within_5pct_of_boundary <- parameters$fraction_of_range <= 0.05 | parameters$fraction_of_range >= 0.95\n    response <- .a4_read(file.path(report, "parameter_response.csv"))\n    metrics$lake <- pairs$lake <- parameters$lake <- response$lake <- lake\n    metric_list[[lake]] <- metrics; pair_list[[lake]] <- pairs\n    param_list[[lake]] <- parameters; response_list[[lake]] <- response\n    timing <- .a4_read(file.path(root, "phase_times.csv")); timing$lake <- lake; timing_list[[lake]] <- timing\n  }\n  metrics <- do.call(rbind, metric_list); pairs <- do.call(rbind, pair_list)\n  parameters <- do.call(rbind, param_list); response <- do.call(rbind, response_list)\n  timing <- do.call(rbind, timing_list)\n  .a4_csv(metrics, file.path(out_dir, "metrics_two_lakes.csv"))\n  .a4_csv(pairs, file.path(out_dir, "matched_window_pairs_two_lakes.csv"))\n  .a4_csv(parameters, file.path(out_dir, "selected_parameters_and_boundaries.csv"))\n  .a4_csv(response, file.path(out_dir, "calibration_parameter_responses.csv"))\n  .a4_csv(timing, file.path(out_dir, "phase_times_all_attempts.csv"))\n  elapsed <- stats::aggregate(timing$elapsed_seconds, timing[c("lake", "phase", "status")], sum)\n  names(elapsed)[ncol(elapsed)] <- "elapsed_seconds"\n  .a4_csv(elapsed, file.path(out_dir, "phase_elapsed_summary.csv"))\n  spans <- do.call(rbind, lapply(lakes, function(lake) {\n    z <- timing[timing$lake == lake, ]; a <- as.POSIXct(z$start_UTC, tz = "UTC"); b <- as.POSIXct(z$end_UTC, tz = "UTC")\n    data.frame(lake = lake, first_phase_start_UTC = format(min(a), "%Y-%m-%d %H:%M:%S", tz = "UTC"),\n      last_phase_end_UTC = format(max(b), "%Y-%m-%d %H:%M:%S", tz = "UTC"),\n      phase_span_seconds = as.numeric(difftime(max(b), min(a), units = "secs")),\n      sum_phase_elapsed_seconds = sum(z$elapsed_seconds), phase_records = nrow(z))\n  }))\n  .a4_csv(spans, file.path(out_dir, "phase_wall_span_summary.csv"))\n  groups <- split(pairs[pairs$split == "validation", ], interaction(pairs$lake[pairs$split == "validation"],\n    pairs$model[pairs$split == "validation"], pairs$month[pairs$split == "validation"], drop = TRUE))\n  monthly <- do.call(rbind, lapply(groups, function(z) cbind(data.frame(lake = z$lake[1], model = z$model[1],\n    window_midpoint_month = z$month[1]), .a4_metrics(z))))\n  .a4_csv(monthly, file.path(out_dir, "monthly_validation_errors.csv"))\n  sim <- response[response$model == "Simstrat", ]\n  triplets <- sim[sim$parameter == "wind_speed", c("lake", "par_id", "status", "RMSE", "value")]\n  names(triplets)[names(triplets) == "value"] <- "wind_speed"\n  for (parameter in c("swr", "ModelParameters/p_sw_ice")) {\n    z <- sim[sim$parameter == parameter, c("lake", "par_id", "value")]\n    names(z)[3] <- if (parameter == "swr") "swr" else "p_sw_ice"\n    triplets <- merge(triplets, z, by = c("lake", "par_id"), all = TRUE)\n  }\n  .a4_need(all(is.finite(triplets$wind_speed) & is.finite(triplets$swr) & is.finite(triplets$p_sw_ice)),\n    "Incomplete Simstrat parameter triplets")\n  .a4_csv(triplets, file.path(out_dir, "simstrat_parameter_triplets.csv"))\n  if (!is.null(old_akesaiqin_metrics)) {\n    old <- .a4_read(old_akesaiqin_metrics)\n    old <- old[old$model %in% models[1:4], fields]\n    comparison <- merge(old, metrics[metrics$lake == "akesaiqin", fields], by = c("model", "split"),\n      suffixes = c("_old_five_model_scheme", "_new_four_model_ice_scheme"))\n    comparison$RMSE_new_minus_old <- comparison$RMSE_new_four_model_ice_scheme - comparison$RMSE_old_five_model_scheme\n    .a4_csv(comparison, file.path(out_dir, "akesaiqin_old_vs_new_individual_models.csv"))\n  }\n  val <- metrics[metrics$split == "validation", ]; val <- val[order(val$lake, val$RMSE), ]\n  val$descriptive_RMSE_rank <- ave(val$RMSE, val$lake, FUN = function(x) rank(x, ties.method = "min"))\n  .a4_csv(val, file.path(out_dir, "validation_comparison.csv"))\n  .a4_figure(out_dir, "01_RMSE_by_lake_and_split", function() {\n    graphics::par(mfrow = c(1, 2), mar = c(7, 5, 3, 1))\n    for (lake in lakes) {\n      z <- metric_list[[lake]]\n      # Explicit indexing avoids assuming report row order.\n      a <- sapply(models, function(m) vapply(c("calibration", "validation"), function(s) z$RMSE[z$model == m & z$split == s], numeric(1)))\n      graphics::barplot(a, beside = TRUE, names.arg = models, las = 2, col = c("#547E9D", "#CF8B4A"),\n        ylab = "Equal-window RMSE (deg C)", main = lake)\n      graphics::legend("topleft", c("Calibration", "Validation"), fill = c("#547E9D", "#CF8B4A"), bty = "n")\n    }\n  })\n  .a4_figure(out_dir, "02_validation_observed_predicted", function() {\n    graphics::par(mfrow = c(2, 5), mar = c(4, 4, 3, 1))\n    for (lake in lakes) for (model in models) {\n      z <- pairs[pairs$lake == lake & pairs$model == model & pairs$split == "validation", ]\n      limits <- range(c(z$observed, z$predicted))\n      graphics::plot(z$observed, z$predicted, xlim = limits, ylim = limits, asp = 1, pch = 16, cex = 0.5,\n        col = grDevices::adjustcolor(colors[model], 0.55), xlab = "Observed window mean (deg C)", ylab = "Predicted window mean (deg C)", main = paste(lake, model))\n      graphics::abline(0, 1, col = "grey40", lty = 2)\n    }\n  }, width = 2300, height = 1050)\n  .a4_figure(out_dir, "03_validation_window_temperature", function() {\n    graphics::par(mfrow = c(2, 1), mar = c(4, 5, 3, 1))\n    for (lake in lakes) {\n      z <- pairs[pairs$lake == lake & pairs$split == "validation", ]\n      graphics::plot(as.Date(z$midpoint_UTC), z$predicted, type = "n", ylim = range(c(z$observed, z$predicted)),\n        xlab = "Window midpoint (UTC)", ylab = "Window mean temperature (deg C)", main = lake)\n      for (model in models) {\n        x <- z[z$model == model, ]; graphics::points(as.Date(x$midpoint_UTC), x$predicted, col = colors[model], pch = 16, cex = 0.45)\n      }\n      x <- z[z$model == models[1], ]; graphics::points(as.Date(x$midpoint_UTC), x$observed, pch = 4, cex = 0.6)\n      graphics::legend("topleft", c("Observed", models), col = c("black", colors), pch = c(4, rep(16, 5)), ncol = 3, cex = 0.7, bty = "n")\n    }\n  })\n  .a4_figure(out_dir, "04_validation_residual_by_window_month", function() {\n    graphics::par(mfrow = c(2, 5), mar = c(4, 4, 3, 1))\n    for (lake in lakes) for (model in models) {\n      z <- pairs[pairs$lake == lake & pairs$model == model & pairs$split == "validation", ]\n      groups <- lapply(1:12, function(m) z$residual[z$month == m])\n      graphics::boxplot(groups, names = 1:12, outline = FALSE, col = colors[model], xlab = "Window midpoint month",\n        ylab = "Predicted - observed (deg C)", main = paste(lake, model))\n      graphics::abline(h = 0, lty = 2, col = "grey40")\n    }\n  }, width = 2300, height = 1050)\n  for (lake in lakes) .a4_figure(out_dir, paste0("05_parameter_responses_", lake), function() {\n    graphics::par(mfrow = c(3, 3), mar = c(4, 4, 3, 1))\n    pars <- param_list[[lake]]\n    for (i in seq_len(nrow(pars))) {\n      z <- response[response$lake == lake & response$model == pars$model[i] & response$parameter == pars$parameter[i] &\n        response$status == "ok" & is.finite(response$RMSE), ]\n      graphics::plot(z$value, z$RMSE, pch = 16, cex = 0.35, col = grDevices::adjustcolor(colors[pars$model[i]], 0.45),\n        xlab = pars$parameter[i], ylab = "Calibration RMSE (deg C)", main = paste(lake, pars$model[i]))\n      graphics::abline(v = pars$value[i], col = "#B51C29", lty = 2)\n    }\n  }, width = 1800, height = 1500)\n  .a4_figure(out_dir, "06_simstrat_joint_parameter_responses", function() {\n    graphics::par(mfrow = c(2, 2), mar = c(4, 4, 3, 1))\n    palette <- c("#254D85", "#3389A5", "#72B9A0", "#DFBD62", "#BD624B")\n    for (lake in lakes) for (parameter in c("wind_speed", "swr")) {\n      z <- triplets[triplets$lake == lake & triplets$status == "ok" & is.finite(triplets$RMSE), ]\n      bin <- pmax(1L, pmin(5L, ceiling(5 * rank(z$RMSE, ties.method = "average")/nrow(z))))\n      graphics::plot(z[[parameter]], z$p_sw_ice, col = grDevices::adjustcolor(palette[bin], 0.65), pch = 16, cex = 0.55,\n        xlab = parameter, ylab = "p_sw_ice", main = paste(lake, "Simstrat"))\n      selected <- param_list[[lake]]; selected <- selected[selected$model == "Simstrat", ]\n      graphics::points(selected$value[selected$parameter == parameter], selected$value[selected$parameter == "ModelParameters/p_sw_ice"], pch = 8, cex = 1.5, lwd = 2)\n      graphics::legend("topleft", legend = c("Lowest 20% RMSE", "20-40%", "40-60%", "60-80%", "Highest 20% RMSE"),\n        col = palette, pch = 16, bty = "n", cex = 0.6)\n    }\n  })\n  note <- c("Two-lake four-model analysis. Report receipt and report-file SHA256 manifests verified.",\n    "Metrics independently reproduced from equal-weight MODIS window pairs. Validation ranking is descriptive; parameter selection uses calibration only.",\n    "Points represent matched window means, not daily observations. No continuous winter series is inferred across missing windows.",\n    "Monthly residuals use window midpoint month. Temperature-proxy ice-free labels are not independent ice observations.",\n    "p_sw_ice=0.9-3 is the requested exploratory range, not an official physical recommendation; simultaneous swr calibration may compensate.",\n    "Parameter-response scatter is not Feldbauer delta/S1 sensitivity analysis. No new engine runs or bootstrap are performed.",\n    "All original run artifacts are read-only. Existing full candidate/native integrity is not rehashed by this plotting script.")\n  writeLines(note, file.path(out_dir, "READ_ME.txt"))\n  cat("TWO_LAKE_ANALYSIS_AND_PLOTS_COMPLETE\\n", out_dir, "\\n", sep = "")\n  invisible(out_dir)\n}\n'
target = local_path(PROJECT) / "analysis_scripts_20261103" / "Analyze_two_lakes_R4.R"
target.parent.mkdir(parents=True, exist_ok=True)
data = R_ANALYSIS.encode("utf-8")
if target.exists() and target.read_bytes() != data:
    raise ValueError(f"目标已有不同代码，保留原文件: {target}")
if not target.exists():
    target.write_bytes(data)
print("TWO_LAKE_ANALYSIS_SCRIPT_READY_NO_MODELS_STARTED")
print("在加载原隔离环境的RStudio会话中运行以下两行：")
r_path = PROJECT + "/analysis_scripts_20261103/Analyze_two_lakes_R4.R"
print("source(" + json.dumps(r_path) + ', encoding = "UTF-8")')
print("ANALYSIS_DIR <- analyze_two_lakes_r4(run_root = " + json.dumps(RUN_ROOT) + ")")


## 在RStudio执行上一个单元打印的命令

使用已加载原隔离环境的RStudio会话。若是全新会话，先source原ISO目录的Restart_LER_window.R，然后执行打印的两行命令。

必须看到`TWO_LAKE_ANALYSIS_AND_PLOTS_COMPLETE`及新输出目录。输出为7幅PNG、7幅PDF及指标/参数/窗口配对/运行阶段耗时CSV。各点代表MODIS匹配窗口均值，不是日观测。

下一单元可单独打包现有正式结果；若分析已经完成，会额外包含最新分析目录。ZIP不包含完整候选工作目录、气象强迫、引擎程序或DLL。


In [ ]:
"""Collect small completed-run evidence without running R or editing any result."""
from pathlib import Path
import csv
import datetime as dt
import hashlib
import io
import json
import os
import zipfile

def local_path(value):
    text = str(value).replace("\\", "/")
    if os.name != "nt" and len(text) >= 3 and text[1:3] == ":/":
        text = "/mnt/" + text[0].lower() + text[2:]
    return Path(text)

def collect_two_lake_results(
    run_root="D:/LER4_ice_20261102/n500",
    config_root="C:/Geography/R/dissertation_LakeEnsemblR/yaml/r4_ice_20261102",
    include_ice=True,
):
    root, config = local_path(run_root), local_path(config_root)
    index_file = root / "two_lake_run_index.csv"
    with index_file.open(encoding="utf-8-sig", newline="") as handle:
        rows = list(csv.DictReader(handle))
    selected = {row["name"]: row for row in rows}
    lakes = ("akesaiqin", "namco")
    if any(name not in selected or selected[name]["calibration_status"] != "ok" or
           selected[name]["evaluation_status"] != "ok" for name in lakes):
        raise ValueError("两湖率定与评价尚未全部完成；未打包。")
    files, skipped = {}, []

    def add(path, archive_name, required=False, max_mb=8):
        if not path.is_file():
            if required:
                raise FileNotFoundError(path)
            return
        if path.stat().st_size > max_mb * 1024**2:
            if required:
                raise ValueError(f"必要文件超过{max_mb}MB，请保留原文件并核对: {path}")
            skipped.append({"file": str(path), "reason": f"larger than {max_mb}MiB"})
            return
        if path.is_symlink():
            raise ValueError(f"不接受符号链接: {path}")
        files[str(archive_name).replace("\\", "/")] = path

    add(index_file, index_file.name, required=True)
    report_names = ("model_metrics.csv", "ensemble4_metrics.csv", "selected_parameters.csv",
                    "parameter_response.csv", "pairs_models_ensemble4.csv", "RMSE_by_split.png", "READ_ME.txt")
    for lake in lakes:
        base = root / lake
        for filename in report_names:
            add(base / "report" / filename, f"{lake}/report/{filename}", required=True)
        add(base / "phase_times.csv", f"{lake}/phase_times.csv", required=True)
        for receipt in ("request.rds", "preflight.rds", "calibration.rds", "evaluation.rds", "report.rds"):
            for ending in ("", ".sha256"):
                name = receipt + ending
                add(base / name, f"{lake}/{name}", required=True)
        for name in ("serial_status.csv", "parallel_status.csv", "workers.log"):
            add(base / "v" / name, f"{lake}/v/{name}")
        small = ("final_metrics.csv", "best_by_calibration_RMSE.csv", "candidate_success_fractions.csv",
                 "calibration_summary.csv", "context.rds", "context_integrity.csv", "metadata.rds",
                 "engine_packages.csv", "function_manifest.csv", "sessionInfo.txt", "workflow_sources.csv",
                 "live_input_files.csv", "workflow_R4ice.R", "recovery_helper_rev3.R")
        for name in small:
            add(base / "c" / name, f"{lake}/c/{name}")
        for model in ("FLake", "GLM", "GOTM", "Simstrat"):
            for prefix in ("params_", "spec_"):
                name = prefix + model + ".csv"
                add(base / "c" / name, f"{lake}/c/{name}", required=True)
        if include_ice:
            sim = base / "c" / "final" / "Simstrat"
            for path in sorted(sim.rglob("*")):
                if not path.is_file():
                    continue
                name = path.name.lower()
                ice = path.suffix.lower() == ".dat" and ("iceh" in name or "snowh" in name)
                metadata = name in {"simstrat.par", "native_config.par", "native_surface.rds", "status.csv",
                                    "contract.csv", "engine_console.log", "warnings.txt"}
                if ice or metadata:
                    add(path, path.relative_to(root).as_posix(), max_mb=5)
    for name in ("selected_lakes.csv", "selected_parameter_design.csv", "_yaml_settings.json",
                 "_puhu_geometry_audit.json", "_runner_sha256.json"):
        add(config / name, "configuration/" + name)
    for lake in lakes:
        name = lake + "_LakeEnsemblR.yaml"
        add(config / name, "configuration/" + name)
    # Include the most recent optional derived analysis, if already created.
    analyses = sorted(p for p in root.parent.glob("analysis_two_lakes_*") if p.is_dir())
    if analyses:
        for path in sorted(analyses[-1].glob("*")):
            if path.is_file():
                add(path, "derived_analysis/" + path.name)
    target_dir = root.parent / "deliveries"
    target_dir.mkdir(parents=True, exist_ok=True)
    stamp = dt.datetime.now().strftime("%Y%m%dT%H%M%S")
    target = target_dir / f"two_lake_results_{stamp}.zip"
    manifest = []
    with zipfile.ZipFile(target, "x", compression=zipfile.ZIP_DEFLATED) as archive:
        for name, path in sorted(files.items()):
            data = path.read_bytes()
            archive.writestr(name, data)
            manifest.append({"file": name, "bytes": len(data), "sha256": hashlib.sha256(data).hexdigest()})
        text = io.StringIO(newline="")
        writer = csv.DictWriter(text, fieldnames=("file", "bytes", "sha256"), lineterminator="\n")
        writer.writeheader(); writer.writerows(manifest)
        archive.writestr("COLLECTION_SHA256.csv", text.getvalue().encode("utf-8-sig"))
        archive.writestr("COLLECTION_NOTES.json", json.dumps({"run_root": str(root), "lakes": lakes,
            "skipped": skipped, "scope": "read-only small result evidence; no forcing, executable, or full candidate folders"},
            ensure_ascii=False, indent=2).encode("utf-8"))
    print("TWO_LAKE_RESULTS_ZIP_READY")
    print(target)
    print(f"{target.stat().st_size / 1024**2:.2f} MB; {len(files)} files")
    return target


RESULTS_ZIP = collect_two_lake_results(RUN_ROOT, CONFIG_ROOT, include_ice=True)
